In [4]:
!pip install bertopic sentence_transformers pyLDAvis mglearn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("Veritasium_comments_sentiment.csv")
print(df.shape)
df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 30.0 MB/s eta 0:00:00
(12899, 15)


,comment_id,author,updated_at,like_count,text,text_normalized,tokens_str,entities_str,entity_texts_str,entity_types_str,keywords_str,vader_score,vader_label,tweetnlp_label,tweetnlp_score
0,0,@DanielRenardAnimation,2019-09-19T18:28:21Z,5751,*Russian Cosmonaut spins a wingnut in space:* ...,russian cosmonaut spins a wingnut in space tel...,russian cosmonaut spins wingnut space tell one,"[('russian', 'NORP')]",russian,NORP,"cosmonaut, russian, tell, wingnut, spins",-0.3595,negative,negative,0.523659
1,1,@NLTops,2020-03-07T07:17:39Z,5244,Someone should tell the Flat Earthers of the D...,someone should tell the flat earthers of the d...,someone tell flat earthers dzhanibekov effect ...,[],NaN,NaN,"ll, sooner, world flip, flat earthers, earthers",0.2023,positive,negative,0.855720
2,2,@aviatordude1961,2021-07-26T02:06:48Z,4005,I thought the reason the Russians kept this a ...,i thought the reason the russians kept this a ...,thought reason russians kept secret going fema...,"[('russians', 'NORP')]",russians,NORP,"win, would always, gold, russians kept, gymnasts",0.6239,positive,neutral,0.696365
3,3,@alvirahesc7436,2021-04-10T12:21:42Z,3842,"""Babe, come over, im home alone""\n\n""No, babe,...","babe, come over, im home alone no, babe, im so...",babe come home alone babe solvin centuries old...,"[('centuries', 'DATE')]",centuries,DATE,"old math, home, centuries, alone, old",-0.5719,negative,neutral,0.720803
4,4,@zeekjones1,2019-09-20T05:46:15Z,2814,Over 300 people broke their phone after watchi...,over 300 people broke their phone after watchi...,300 people broke phone watching video,"[('over 300', 'CARDINAL')]",over 300,CARDINAL,"300, phone watching, broke, watching video, wa...",-0.4215,negative,negative,0.834433


In [7]:
%%time
import numpy as np
# vectorize using tokens_str (already preprocessed)
# Fill NaN values with empty strings before vectorization
df['tokens_str'] = df['tokens_str'].fillna('')
vect = CountVectorizer(max_features=1000, max_df=.5)
X = vect.fit_transform(df['tokens_str'])

# train LDA with 8 topics
lda = LatentDirichletAllocation(n_components=10, learning_method="batch",
                                max_iter=25, random_state=0)

document_topics = lda.fit_transform(X)

print("lda.components_.shape: {}".format(lda.components_.shape))

# show top words per topic
sorting = np.argsort(lda.components_, axis=1)[:, ::-1]
feature_names = np.array(vect.get_feature_names_out())

print(feature_names[sorting])

lda.components_.shape: (10, 1000)
[['would' 'energy' 'like' ... 'claw' 'lands' 'skateboarding']
 ['flip' 'phone' 'tennis' ... 'caps' 'baby' 'melt']
 ['force' 'centrifugal' 'forces' ... 'knife' 'melts' 'football']
 ...
 ['earth' 'mass' 'change' ... 'phone' 'skateboard' 'earthers']
 ['video' 'great' 'thanks' ... 'elevator' 'shifted' 'galaxy']
 ['axis' 'inertia' 'moment' ... 'passwords' 'table' 'claw']]
CPU times: user 1min, sys: 62.6 ms, total: 1min
Wall time: 1min 1s


In [9]:
import mglearn
mglearn.tools.print_topics(topics=range(10), feature_names=feature_names,
                           sorting=sorting, topics_per_chunk=4, n_words=10)

topic 0       topic 1       topic 2       topic 3       
--------      --------      --------      --------      
would         flip          force         explanation   
energy        phone         centrifugal   intuitive     
like          tennis        forces        explain       
gravity       flipping      math          effect        
momentum      always        motion        physics       
even          racket        last          video         
spinning      years         rotating      dzhanibekov   
angular       noticed       like          understand    
kinetic       one           makes         axis          
ball          hammer        saying        could         


topic 4       topic 5       topic 6       topic 7       
--------      --------      --------      --------      
earth         spin          earth         earth         
flip          axis          flip          mass          
moon          object        magnetic      change        
flipped       flip          s

In [10]:
df['lda_topic'] = np.argmax(document_topics, axis=1)
print(df['lda_topic'].value_counts())
df[['text_normalized','lda_topic']].head(10)

lda_topic
8    2592
1    1601
7    1559
5    1394
6    1346
4    1131
3     983
0     964
9     734
2     595
Name: count, dtype: int64


,text_normalized,lda_topic
0,russian cosmonaut spins a wingnut in space tel...,5
1,someone should tell the flat earthers of the d...,1
2,i thought the reason the russians kept this a ...,1
3,"babe, come over, im home alone no, babe, im so...",2
4,over 300 people broke their phone after watchi...,8
5,oh! so thats why my liquid filled bullets keep...,6
6,this explaination is beautiful when you're act...,8
7,750 australians didnt like the fact earth wont...,6
8,video contains the phrase prove feynman wrong ...,8
9,"as a kid, i would frequently watch my dad flip...",8


In [11]:
import pyLDAvis
import pyLDAvis.lda_model
pyLDAvis.enable_notebook()

pyLDAvis.lda_model.prepare(lda, X, vect)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5     -0.035389  0.096889       1        1  14.380388
7     -0.208329 -0.011805       2        1  13.292519
8      0.250088 -0.125861       3        1  12.675713
1      0.065463 -0.158085       4        1  10.859972
9     -0.099368  0.289411       5        1   9.660553
6     -0.183113 -0.183789       6        1   9.196495
0     -0.013668  0.054145       7        1   8.553091
4     -0.151577 -0.091604       8        1   8.286942
3      0.185201  0.030894       9        1   7.350945
2      0.190690  0.099806      10        1   5.743382, topic_info=        Term         Freq        Total Category  logprob  loglift
238    earth  4265.000000  4265.000000  Default  30.0000  30.0000
74      axis  2537.000000  2537.000000  Default  29.0000  29.0000
417  inertia  1129.000000  1129.000000  Default  28.0000  28.0000
547   moment   861.000000   861.000000  Default  27.0000  27.0000
946    video  1407.000000  1407.000000  Default  26.0000  26.0000
..       ...          ...          ...      ...      ...      ...
882    thing    65.835238   394.644278  Topic10  -4.7890   1.0663
280  explain    63.105993   438.863371  Topic10  -4.8313   0.9177
813    space    68.498850   813.239830  Topic10  -4.7493   0.3829
76      back    58.498576   354.335078  Topic10  -4.9071   1.0559
884    think    57.799702   656.921610  Topic10  -4.9191   0.4265

[580 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
0         6  0.992919      000
1         6  0.806702      100
1         7  0.177081      100
3         1  0.134295      1st
3         5  0.850535      1st
...     ...       ...      ...
995       8  0.076456    years
998       3  0.986273  youtube
999       1  0.403836     zero
999       7  0.527459     zero
999      10  0.065932     zero

[1272 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[6, 8, 9, 2, 10, 7, 1, 5, 4, 3])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


For bert I changed runtime type to GPU

In [2]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.0 MB/s eta 0:00:00


In [3]:
!pip install sentence_transformers

In [7]:
import pandas as pd

df = pd.read_csv("Veritasium_comments_sentiment.csv")
print(df.shape)
df.head()

(12899, 15)


,comment_id,author,updated_at,like_count,text,text_normalized,tokens_str,entities_str,entity_texts_str,entity_types_str,keywords_str,vader_score,vader_label,tweetnlp_label,tweetnlp_score
0,0,@DanielRenardAnimation,2019-09-19T18:28:21Z,5751,*Russian Cosmonaut spins a wingnut in space:* ...,russian cosmonaut spins a wingnut in space tel...,russian cosmonaut spins wingnut space tell one,"[('russian', 'NORP')]",russian,NORP,"cosmonaut, russian, tell, wingnut, spins",-0.3595,negative,negative,0.523659
1,1,@NLTops,2020-03-07T07:17:39Z,5244,Someone should tell the Flat Earthers of the D...,someone should tell the flat earthers of the d...,someone tell flat earthers dzhanibekov effect ...,[],NaN,NaN,"ll, sooner, world flip, flat earthers, earthers",0.2023,positive,negative,0.855720
2,2,@aviatordude1961,2021-07-26T02:06:48Z,4005,I thought the reason the Russians kept this a ...,i thought the reason the russians kept this a ...,thought reason russians kept secret going fema...,"[('russians', 'NORP')]",russians,NORP,"win, would always, gold, russians kept, gymnasts",0.6239,positive,neutral,0.696365
3,3,@alvirahesc7436,2021-04-10T12:21:42Z,3842,"""Babe, come over, im home alone""\n\n""No, babe,...","babe, come over, im home alone no, babe, im so...",babe come home alone babe solvin centuries old...,"[('centuries', 'DATE')]",centuries,DATE,"old math, home, centuries, alone, old",-0.5719,negative,neutral,0.720803
4,4,@zeekjones1,2019-09-20T05:46:15Z,2814,Over 300 people broke their phone after watchi...,over 300 people broke their phone after watchi...,300 people broke phone watching video,"[('over 300', 'CARDINAL')]",over 300,CARDINAL,"300, phone watching, broke, watching video, wa...",-0.4215,negative,negative,0.834433


In [9]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

topic_model_Bert = BERTopic(embedding_model="xlm-r-bert-base-nli-stsb-mean-tokens")

# fix: convert all to string and drop empty
docs = df['tokens_str'].fillna('').astype(str).tolist()
docs = [d if d.strip() != '' else 'empty' for d in docs]

bert_topics, _ = topic_model_Bert.fit_transform(docs)

df['bertopic_topic'] = bert_topics

freq = topic_model_Bert.get_topic_info()
print("Number of topics: {}".format(len(freq)))
freq.head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Number of topics: 164


,Topic,Count,Name,Representation,Representative_Docs
0,-1,5790,-1_axis_would_earth_like,"[axis, would, earth, like, flip, rotation, spi...",[happens start spinning object axis greatest m...
1,0,463,0_ice_caps_melting_melt,"[ice, caps, melting, melt, poles, polar, chang...","[mean melting polar ice caps may flip planet, ..."
2,1,373,1_tennis_racket_racquet_without,"[tennis, racket, racquet, without, rackets, pe...",[tennis racket spin using machine flip tennis ...
3,2,213,2_wing_nut_wingnut_wings,"[wing, nut, wingnut, wings, aircraft, pitch, c...",[well let see free fall space station observe ...
4,3,202,3_hammer_claw_hammers_head,"[hammer, claw, hammers, head, noticed, handle,...",[used claw hammer years ago kid n't know geniu...
5,4,200,4_planets_planet_moon_core,"[planets, planet, moon, core, sun, earth, magn...",[got something wrong wrong sphere exitbit effe...
6,5,189,5_voice_johnny_option_veritasium,"[voice, johnny, option, veritasium, paper, wat...",[deduced edgar cayce readings several person p...
7,6,123,6_video_best_videos_seen,"[video, best, videos, seen, great, awesome, go...","[one best videos 've done awesome, best video ..."
8,7,115,7_flip_earth_happen_flipped,"[flip, earth, happen, flipped, tha, wanted, ho...","[earth flip, earth flip, earth flip]"
9,8,115,8_food_blushing_regret_syntactically,"[food, blushing, regret, syntactically, advise...",[green grey grieving fairies developmentally h...


In [10]:
# reduce to more manageable number of topics
topic_model_Bert.reduce_topics(docs, nr_topics=20)

freq = topic_model_Bert.get_topic_info()
print("Number of topics: {}".format(len(freq)))
freq.head(20)

Number of topics: 20


,Topic,Count,Name,Representation,Representative_Docs
0,-1,5790,-1_earth_axis_would_flip,"[earth, axis, would, flip, like, video, rotati...",[n't see 'intuitive explanation would differen...
1,0,3719,0_earth_flip_axis_would,"[earth, flip, axis, would, ice, mass, inertia,...",[earth n't flip claim interesting firm believe...
2,1,948,1_video_flat_thank_thanks,"[video, flat, thank, thanks, great, explanatio...","[great video thank, thank great video, great v..."
3,2,856,2_video_physics_passwords_secret,"[video, physics, passwords, secret, password, ...",[thanks lot posting passwords video speaking v...
4,3,267,3_hammer_claw_hammers_football,"[hammer, claw, hammers, football, noticed, hea...",[used claw hammer years ago kid n't know geniu...
5,4,246,4_feynman_wrong_prove_goal,"[feynman, wrong, prove, goal, right, richard, ...","[goal video prove feynman wrong, goal video pr..."
6,5,192,5_phone_else_phones_flipping,"[phone, else, phones, flipping, flipped, tried...","[flipped phone times watching video phone, pho..."
7,6,165,6_west_sun_rise_day,"[west, sun, rise, day, quran, islam, prophet, ...","[prophet said sun rise west, quran says end ti..."
8,7,157,7_baby_bottle_space_bottles,"[baby, bottle, space, bottles, station, iss, b...","[baby bottle space, baby bottle space, baby bo..."
9,8,115,8_skateboard_impossible_skateboarding_gymnasts,"[skateboard, impossible, skateboarding, gymnas...",[skateboard trick called impossible called imp...


In [11]:
topic_model_Bert.visualize_barchart(top_n_topics=10)

In [12]:
topic_model_Bert.visualize_hierarchy(top_n_topics=15)

In [13]:
for sentiment in ['positive', 'negative', 'neutral']:
    subset = df[df['tweetnlp_label'] == sentiment]['tokens_str'].fillna('').astype(str).tolist()
    subset = [d if d.strip() != '' else 'empty' for d in subset]
    if len(subset) < 50:
        continue
    model = BERTopic(embedding_model="xlm-r-bert-base-nli-stsb-mean-tokens", verbose=False)
    topics, _ = model.fit_transform(subset)
    print(f"\n=== Top topics for {sentiment.upper()} comments ===")
    print(model.get_topic_info().head(8))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Top topics for POSITIVE comments ===
   Topic  Count                                Name  \
0     -1    797   -1_video_great_explanation_videos   
1      0    301         0_earth_axis_would_rotation   
2      1    115          1_video_thank_great_thanks   
3      2    102       2_thank_thanks_sharing_relief   
4      3     63          3_video_never_slow_youtube   
5      4     58     4_tennis_racket_racquet_rackets   
6      5     58        5_awesome_australia_phew_dam   
7      6     53  6_math_maths_mathematician_teacher   

                                      Representation  \
0  [video, great, explanation, videos, thanks, lo...   
1  [earth, axis, would, rotation, inertia, effect...   
2  [video, thank, great, thanks, excellent, expla...   
3  [thank, thanks, sharing, relief, brilliant, in...   
4  [video, never, slow, youtube, yall, watch, vid...   
5  [tennis, racket, racquet, rackets, ve, flip, w...   
6  [awesome, australia, phew, dam, up, one, wow, ...   
7  [math, math

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Top topics for NEGATIVE comments ===
   Topic  Count                         Name  \
0     -1      1   -1_round_right_ads_youtube   
1      0   2522      0_earth_flip_would_like   
2      1     27  1_baby_space_bottle_station   
3      2     26  2_baby_bottle_space_bottles   

                                      Representation  \
0  [round, right, ads, youtube, head, baby, spinn...   
1  [earth, flip, would, like, video, axis, one, e...   
2  [baby, space, bottle, station, bottles, babies...   
3  [baby, bottle, space, bottles, question, would...   

                                 Representative_Docs  
0  [youtube ads spinning head right round baby ri...  
1  [explanation effect explanation earth going fl...  
2  [baby bottle space station, baby bottle space ...  
3  [baby bottle, would baby bottle space, baby bo...  


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Top topics for NEUTRAL comments ===
   Topic  Count                            Name  \
0     -1   3164        -1_axis_would_earth_flip   
1      0    428         0_ice_caps_melting_melt   
2      1    257  1_tennis_racket_racquet_always   
3      2    195         2_planet_core_earth_sun   
4      3    189    3_phone_else_watching_phones   
5      4    159        4_wing_nut_wingnut_wings   
6      5    124     5_earth_flipped_happen_flip   
7      6    109  6_voice_elon_oumuamua_tomorrow   

                                      Representation  \
0  [axis, would, earth, flip, object, spinning, l...   
1  [ice, caps, melting, melt, poles, polar, mass,...   
2  [tennis, racket, racquet, always, without, per...   
3  [planet, core, earth, sun, planets, magnetic, ...   
4  [phone, else, watching, phones, flipping, trie...   
5  [wing, nut, wingnut, wings, aircraft, pitch, c...   
6  [earth, flipped, happen, flip, past, would, th...   
7  [voice, elon, oumuamua, tomorrow, musk, famili..

In [14]:
df.to_csv("Veritasium_comments_modeling_topics.csv", index=False)
print(f"Saved {len(df)} comments with topics")

Saved 12899 comments with topics
